# COGS 108 - EDA Checkpoint

## Authors

Team list and credits:
- Katelyn Yap: Conceptualization, Data set 1, Writing - code cleanup
- Vy Nguyen: Ethics Revision, Project Timeline Proposal, Dataset 3
- Ria Parikh: Dataset 4, Writing - code cleanup
- Laird Fowle: Dataset 2, Writing - code cleanup
- Cindy Tao: Dataset 5, Writing - code cleanup


# Research Question

To what extent does variation in daily Air Quality Index (AQI), as measured by the U.S. Environmental Protection Agency (EPA), explain variation in daily visitor trailhead sensor counts in Yosemite National Park between 1990 and 2025, controlling for daily maximum temperature and total precipitation?


## Background and Prior Work

## Background and Prior Work

Outdoor recreation in national parks is influenced by environmental conditions that affect both visitor safety and overall experience. In the western United States, air quality has become an increasingly important factor due to wildfire smoke and regional pollution. The U.S. Environmental Protection Agency’s Air Quality Index (AQI) is a standardized measure used to communicate the health risks associated with air pollution and to inform the public about outdoor activity safety.

Because many national park activities involve extended outdoor physical activity, changes in air quality may affect visitors’ decisions about whether and when to visit. Understanding how air quality relates to park visitation is therefore relevant for park management, public health communication, and planning in highly visited parks such as Yosemite National Park.

Previous research has established a clear connection between environmental conditions and tourism behavior. Buckley (2011) reviews existing literature on tourism and the environment and finds that weather, climate variability, and environmental quality consistently influence visitation patterns in natural areas. <a href="#ref1">1</a> The review also notes that visitor responses differ depending on the type of destination and visitor motivations, suggesting that environmental effects may not be uniform across parks. However, Buckley’s work is conceptual and broad in scope, leaving open questions about how specific environmental indicators, such as AQI, influence visitation at individual parks.

Empirical studies provide further evidence that air pollution affects park visitation. Keiser, Lade, and Rudik (2018) examine monthly visitation data from 33 U.S. national parks and find that higher ozone pollution levels are associated with reduced visitation, even after accounting for weather, seasonality, and park-specific differences. <a href="#ref2">2</a> While this study demonstrates that air pollution can influence park use at a national scale, it relies on monthly averages and does not isolate park-specific daily behavioral responses. In addition, although Yosemite is included in their broader sample, the study does not focus specifically on Yosemite’s unique visitation patterns or wildfire-related air quality variability.

More recent work has used alternative data sources to study visitation patterns at finer time scales. Minehart et al. (2025) analyze weekly visitation to parks and protected areas in the Pacific Northwest using anonymized mobile device data, examining the effects of temperature, precipitation, AQI, and particulate matter (PM2.5). <a href="#ref3">3</a> The authors find that higher levels of PM2.5 are generally associated with lower visitation, while the relationship between AQI and visitation varies across different types of parks. However, their analysis aggregates across multiple parks and regions, making it difficult to determine how daily air quality fluctuations affect a single, high-profile destination over a long time horizon.

Together, these studies suggest that environmental quality and weather conditions play an important role in shaping park visitation, while also highlighting important gaps in the literature. Much of the prior work relies on monthly or weekly data and emphasizes cross-park comparisons rather than park-specific, daily analysis. Furthermore, limited attention has been given to how short-term variation in AQI influences visitation in an iconic and highly visited park such as Yosemite, particularly over multiple decades that include periods of increasing wildfire activity.

The present project addresses these gaps by examining whether daily variation in AQI predicts daily visitation to Yosemite National Park from 1979 to 2023, while controlling for daily maximum temperature and total precipitation. By focusing on a single park and using daily data, this study aims to capture short-term behavioral responses to air quality conditions that may be obscured in broader, aggregated analyses. This park-specific approach provides clearer insight into how air pollution influences visitation decisions in one of the most prominent national parks in the United States.

1. <a name="ref1"></a> Buckley, R. (2011). Tourism and environment. Annual Review of Environment and Resources, 36, 397–416. https://www.annualreviews.org/content/journals/10.1146/annurev-environ-041210-132637

2. <a name="ref2"></a> Keiser, D., Lade, G. E., & Rudik, I. (2018). Air pollution and visitation at U.S. national parks. Science Advances, 4(7). https://pmc.ncbi.nlm.nih.gov/articles/PMC6051738/

3. <a name="ref3"></a> Minehart, J., et al. (2025). The mountains are calling, but will visitors go? Modeling the effect of weather and air quality on visitation to Pacific Northwest parks and protected areas using mobile device data. https://www.researchgate.net/publication/390630027_The_mountains_are_calling_but_will_visitors_go_Modeling_the_effect_of_weather_and_air_quality_on_visitation_to_Pacific_Northwest_parks_and_protected_areas_using_mobile_device_data


# Hypothesis


We hypothesize that there is a statistically significant negative correlation between the daily Air Quality Index (AQI) and daily visitor trailhead sensor counts in Yosemite National Park from 1999–2025. Specifically, we predict that as the AQI value increases (indicating worse air quality), the number of daily trail users will decrease, and this relationship will continue even after controlling for daily maximum temperature and total precipitation.


## Data

### Data overview

- **Dataset #1:** EPA Daily Air Quality Index — Mariposa County, CA
  - Link: https://www.epa.gov/outdoor-air-quality-data/download-daily-data
  - ~one row per monitoring day; 34 years (1990–2023) of data.
  - Key variables: Date, AQI, Category, Defining Parameter.
  - Shortcomings: Sparse pre-1990 data; spatial mismatch between Yosemite Village monitor and remote trailheads; missing days must be treated as genuinely missing.

- **Dataset #2:** NPS Yosemite Visitor Count Data
  - Link: https://irma.nps.gov/Stats/SSRSReports/Park%20Specific%20Reports/Recreation%20Visitors%20By%20Month%20(1979%20-%20Last%20Calendar%20Year)?Park=YOSE
  - 564 monthly observations (1979–2025). Key variable: Visitors.
  - Shortcomings: Monthly granularity only; does not capture visitors entering outside official entrances.


In [ ]:
# Run this code every time when you're actively developing modules in .py files.
%load_ext autoreload
%autoreload 2


In [ ]:
# Setup code -- Run only once after cloning!!!
# Downloads raw AQI zip files for each year to data/00-raw/

%pip install requests tqdm

import sys
sys.path.append('./modules')
import get_data

datafiles = [
    {'url': f'https://aqs.epa.gov/aqsweb/airdata/daily_aqi_by_county_{year}.zip',
     'filename': f'daily_aqi_by_county_{year}.zip'}
    for year in range(1990, 2024)
]

get_data.get_raw(datafiles, destination_directory='data/00-raw/')


### Dataset #1: EPA Daily Air Quality Index — Mariposa County, CA

The EPA Daily AQI dataset provides one composite air quality value per day for Mariposa County, California, derived from ground-level monitor readings. The Air Quality Index (AQI) is a unitless scale from 0 to 500: values 0–50 are 'Good,' 51–100 are 'Moderate,' 101–150 are 'Unhealthy for Sensitive Groups,' 151–200 are 'Unhealthy,' 201–300 are 'Very Unhealthy,' and above 300 is 'Hazardous.'

The Defining Parameter column identifies which pollutant drove the AQI value that day — in the Yosemite area this is most commonly PM2.5 (fine particulate matter from wildfire smoke) during summer and fall, or ground-level Ozone during hot weather periods.

Key concerns: the data only goes back to 1990; the monitor is at Yosemite Village (a single point in a very large park); missing days due to monitor downtime must be treated as genuinely missing, not 'Good' air quality; and the county-level summary takes the maximum AQI across all monitors in Mariposa County.


In [ ]:
import zipfile
import pandas as pd
import os

# Load and filter all years to Mariposa County
dfs = []
for year in range(1990, 2024):
    path = f'data/00-raw/daily_aqi_by_county_{year}.zip'
    try:
        with zipfile.ZipFile(path, 'r') as z:
            fname = z.namelist()[0]
            with z.open(fname) as f:
                df = pd.read_csv(f)
                mariposa = df[df['county Name'] == 'Mariposa'].copy()
                dfs.append(mariposa)
    except Exception as e:
        print(f'Could not load {year}: {e}')

aqi = pd.concat(dfs, ignore_index=True)
aqi['Date'] = pd.to_datetime(aqi['Date'])

# 1. Size
print(f'Shape: {aqi.shape}')
print(f'Date range: {aqi["Date"].min()} to {aqi["Date"].max()}')

# 2. Duplicates
print(f'\nDuplicate dates: {aqi["Date"].duplicated().sum()}')

# 3. Missingness
print('\nMissing values per column:')
print(aqi.isnull().sum())

# 4. Missing calendar days
full_range = pd.date_range(start='1990-01-01', end='2023-12-31')
missing_days = full_range.difference(aqi['Date'])
print(f'\nCalendar days with no AQI reading: {len(missing_days)}')
print('Missing days are dropped — a missing reading does not mean Good air quality.')

# 5. Outliers
print('\nAQI summary statistics:')
print(aqi['AQI'].describe())
outliers = aqi[aqi['AQI'] > 300]
print(f'\nDays with AQI > 300 (Hazardous): {len(outliers)}')
print(outliers[['Date', 'AQI', 'Category', 'Defining Parameter']])

# 6. Defining parameter frequency
print('\nDefining Parameter frequency:')
print(aqi['Defining Parameter'].value_counts())

# 7. Save
os.makedirs('data/02-processed', exist_ok=True)
aqi.to_csv('data/02-processed/mariposa_aqi.csv', index=False)
print('\nSaved to data/02-processed/mariposa_aqi.csv')


### Dataset #2: NPS Yosemite Monthly Visitor Count Data

The Yosemite visitation dataset provided by the National Park Service (NPS) provides monthly visitation data for the park spanning 1979–2025 with 564 observations. This data provides our experiment with its key dependent variable, visitor count, separated by year and month. Data is collected through counting park admittances at Yosemite's 5 entrance gates.

The main concern with this dataset is the lack of daily visitation data — the NPS only provides data at a monthly level, meaning trends will need to be observed over a longer time frame.


In [ ]:
import pandas as pd
import io
import requests
import os

# Load data
url = ('https://raw.githubusercontent.com/COGS108/Group046_WI26/72efcfcfe6ea229d895a0e93db5f02fc74ef297c'
       '/data/00-raw/Recreation%20Visitors%20By%20Month%20(1979%20-%20Last%20Calendar%20Year).csv')
s = requests.get(url).content

# Remove non-data rows and columns
df = pd.read_csv(io.StringIO(s.decode('utf-8')), skiprows=2)
df = df.drop(columns=['Textbox5'])

# Wrangle month columns
month_cols = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
for col in month_cols:
    df[col] = df[col].astype(str).str.replace(',','').replace('nan','0')
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Melt wide to long
df_tidy = df.melt(id_vars=['Year'], value_vars=month_cols,
                  var_name='Month', value_name='Visitors')

month_map = {'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,
             'JUL':7,'AUG':8,'SEP':9,'OCT':10,'NOV':11,'DEC':12}
df_tidy['Month_Num'] = df_tidy['Month'].map(month_map)
df_tidy['Date'] = pd.to_datetime(df_tidy[['Year']].assign(
    Month=df_tidy['Month_Num'], Day=1))
df_tidy = df_tidy[df_tidy['Year'] != 2026].copy()

# Diagnostics
print(f'Shape: {df_tidy.shape}')
print(f'Date range: {df_tidy["Date"].min().date()} to {df_tidy["Date"].max().date()}')
print(f'\nDuplicate month/year combos: {df_tidy[["Year","Month"]].duplicated().sum()}')
print('\nMissing values per column:')
print(df_tidy.isnull().sum())
print('\nVisitor count summary statistics:')
print(df_tidy['Visitors'].describe())

# Save
os.makedirs('data/02-processed', exist_ok=True)
df_tidy.to_csv('data/02-processed/yosemite_visitors_cleaned.csv', index=False)
print('\nSaved to data/02-processed/yosemite_visitors_cleaned.csv')
df_tidy = df_tidy.sort_values('Date').reset_index(drop=True)
print(df_tidy.head())


## Results

### Exploratory Data Analysis

This section explores Dataset #1 — the EPA Daily AQI data for Mariposa County, CA (1990–2023). We load the fully wrangled data directly from `data/02-processed/mariposa_aqi.csv` so this section can be run independently of the data-download cells above. The goal is to understand how air quality has changed over time, which is the foundational environmental story our research question is built on.


### Annual Average AQI in Mariposa County (1990–2023)

The plot below shows the annual mean AQI for Mariposa County over the 34-year study period. This is the most direct way to visualize whether air quality has changed over time. If average AQI has trended upward (worse air quality), it supports the premise that wildfire smoke has become an increasingly relevant factor in visitor decision-making. We use a scatter plot with a linear trend line overlaid to show both year-to-year variability and the long-run direction. Notable spike years (e.g., 2020, 2021) are expected to correspond to major wildfire seasons in the Sierra Nevada region.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

# Load processed AQI data
aqi = pd.read_csv('data/02-processed/mariposa_aqi.csv', parse_dates=['Date'])
aqi['Year'] = aqi['Date'].dt.year
annual_mean = aqi.groupby('Year')['AQI'].mean().reset_index()

# Fit a linear trend line
z = np.polyfit(annual_mean['Year'], annual_mean['AQI'], 1)
p = np.poly1d(z)

# AQI category background bands
category_bands = [
    (0,   50,  '#00e400', 'Good'),
    (51,  100, '#ffff00', 'Moderate'),
    (101, 150, '#ff7e00', 'Unhealthy for Sensitive Groups'),
]

fig, ax = plt.subplots(figsize=(12, 5))

for lo, hi, color, label in category_bands:
    ax.axhspan(lo, hi, alpha=0.12, color=color, label=label)

# Scatter points
ax.scatter(annual_mean['Year'], annual_mean['AQI'],
           color='#2c7bb6', s=60, zorder=3, label='Annual Mean AQI')

# Trend line
x_range = np.linspace(annual_mean['Year'].min(), annual_mean['Year'].max(), 200)
ax.plot(x_range, p(x_range), color='#d7191c', linewidth=2,
        linestyle='--', label=f'Trend (slope={z[0]:.2f}/yr)')

# Annotate worst year
worst = annual_mean.loc[annual_mean['AQI'].idxmax()]
ax.annotate(f"{int(worst['Year'])}\n(AQI={worst['AQI']:.0f})",
            xy=(worst['Year'], worst['AQI']),
            xytext=(worst['Year'] - 3, worst['AQI'] + 3),
            fontsize=9, color='#d7191c',
            arrowprops=dict(arrowstyle='->', color='#d7191c'))

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Mean AQI', fontsize=12)
ax.set_title('Annual Average Air Quality Index — Mariposa County, CA (1990–2023)',
             fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
ax.legend(loc='upper left', fontsize=9)
ax.set_ylim(0, annual_mean['AQI'].max() + 10)
plt.tight_layout()
plt.savefig('data/02-processed/aqi_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Overall trend: {z[0]:+.3f} AQI units per year')
print(f'Worst year:  {int(worst["Year"])} (mean AQI = {worst["AQI"]:.1f})')
print(f'Best year:   {int(annual_mean.loc[annual_mean["AQI"].idxmin(), "Year"])} '
      f'(mean AQI = {annual_mean["AQI"].min():.1f})')


The trend line slope tells us the average annual change in AQI over the study period. A positive slope indicates worsening air quality over time, consistent with increased wildfire activity in the Sierra Nevada. Spike years (typically post-2015) are expected to align with well-documented major fire seasons such as the 2018 Camp Fire and the 2020–2021 fire seasons. This upward trend strengthens the rationale for studying how AQI affects visitor behavior, as deteriorating air quality becomes an increasingly relevant factor for park management and public health communication.


## Ethics

### A. Data Collection
 - **A.1 Informed consent**: Our project utilizes secondary data from trailhead sensors and EPA monitors. The data is collected at an aggregate level with no Personally Identifiable Information (PII). We treat missing monitor days as genuinely missing rather than 'Good' air quality.

 - [X] **A.2 Collection bias**: We acknowledge that activity outside official park hours or uncharted areas of the park is not captured. We focus on consistent data sources and specify temporal scope to avoid confounding variables.
 - [X] **A.3 Limit PII exposure**: All data is anonymized aggregate counts. We limit the dataset to only variables necessary to address the research question.

 - **A.4 Downstream bias mitigation**: Results cannot be used to infer behavioral differences among specific groups of people.

### B. Data Storage
 - [ ] **B.1 Data security**: Data is stored on password-protected devices and a private GitHub repository accessible only by team members.
 - [ ] **B.2 Right to be forgotten**: Data is fully anonymized; no individual records exist to remove.
 - [ ] **B.3 Data retention plan**: Datasets will be deleted after the course project unless needed for academic purposes.

### C. Analysis
 - [ ] **C.1 Missing perspectives**: We cannot account for individual visitor motivations, health conditions, or socioeconomic constraints. We ground interpretations in established literature.
 - [ ] **C.2 Dataset bias**: Spatial mismatch between the Yosemite Village EPA monitor and high-elevation trailheads may not reflect localized smoke patterns. We mitigate this with temperature and precipitation control variables.
 - [ ] **C.3 Honest representation**: Results will be conveyed as associations, not causal relationships. All figures will have clearly labeled axes and 95% confidence intervals.
 - [ ] **C.4 Privacy in analysis**: Only aggregated daily-level summaries are reported.
 - [ ] **C.5 Auditability**: Full workflow is maintained in a GitHub repository with version control.

### D. Modeling
 - [X] **D.1 Proxy discrimination**: Trailhead counts may proxy for socioeconomic status. We will acknowledge that the average behavioral response may be skewed toward visitors with more autonomy to reschedule trips.
 - [X] **D.2 Fairness across groups**: AQI disproportionately impacts vulnerable groups. Stable attendance numbers do not imply all visitor groups are equally protected.
 - [X] **D.3 Metric selection**: Metrics are framed as indicators of recreational demand during environmental hazards, not holistic measures of trail importance.
 - [X] **D.4 Explainability**: Multiple linear regression coefficients and 95% confidence intervals will be reported to ensure interpretable results for stakeholders.
 - [X] **D.5 Communicate limitations**: Study is restricted to Yosemite, 1990–2023, and findings may not generalize to other parks. The model identifies associations, not causation.

### E. Deployment
 - [X] **E.1 Monitoring and evaluation**: We will audit predictions against actual sensor counts to detect concept drift over time.
 - [X] **E.2 Redress**: If outputs lead to societal harm or resource misallocation, we will review and correct the model.
 - [X] **E.3 Roll back**: Git version control allows rollback to any stable state if a prediction creates a problematic feedback cycle.
 - [X] **E.4 Unintended use**: We will clearly communicate that the model identifies associations rather than causal relationships to prevent misuse.


## Team Expectations

Katelyn Yap's Expectations: Respond to group chat messages within 10 hours. Let members know when they are going to be too busy (midterms etc).

Vy Nguyen's Expectations: Communications on Discord will be replied to during school and wake hours 9am–10pm. Communication on progress will be updated every week during our meeting (online or in-person).

Laird Fowle's Expectations: There will be weekly meetings that discuss deadlines/obligations to ensure everyone is up to date. Max response wait of 2 hours, especially on the day of a deadline.

Cindy Tao's Expectations: Everyone will complete assigned tasks on time and communicate ahead of the deadline if issues arise. Feedback and disagreements will be handled respectfully and constructively.

Ria Parikh's Expectations: There will be weekly check-ins on assigned tasks. Ensure to respond in a timely manner in the Discord chat. Address any absences or potential sickness/lateness prior to due dates.


## Project Timeline Proposal

| Date & Time        | Task |
|-------------------|------|
| 2/18, 11:00 PM    | Data Curation: Search for 'gettable' datasets. Checkpoint 1: Review data wrangling; ensure data is Tidy and Clean. |
| 3/4, 11:00 PM     | Visualization: Generate univariate and bivariate plots for EDA. Checkpoint 2: Discuss patterns discovered in EDA; refine modeling strategy. |
| 3/13, 11:00 PM    | Analysis: Finalize inferential or predictive models (e.g., Regression, ANOVA). Draft results, discussion, and conclusion sections of the final report. |
| 3/18, 11:00 PM    | Record Final Video summary for a non-technical audience. Turn in Final Report and Video; complete Group Project Surveys. |
